# SQLite experimentation

In [3]:
import sqlite3
from pathlib import Path 
db_fp = Path(r'C:\Users\oli_n\pycode\examserver\DB\chimera_server.db')
# db_fp = Path(r'C:\Users\oli_n\pycode\examserver\DB\onstartup_test_server.db')

try:
    connection = sqlite3.connect(db_fp)
    cursor = connection.cursor()
    print(f'***Opened DB {db_fp} ***')
except Exception as e:
    print('FAILED....')
    print(e)


def execute_query(self, query, txt=''):
    cursor.execute(query)
    connection.commit()
    print(f"\t[DB SExecute] '{txt}' Query successful")



***Opened DB C:\Users\oli_n\pycode\examserver\DB\chimera_server.db ***


In [7]:
def get_rr_case(uid, case_n):
        '''
        Specific single RR case lookup
        return the case as json
        '''
        query = '''
        SELECT case_number, rr_normal, rr_abnormal, rr_desc 
        FROM rr_answers
        WHERE uid=? AND case_number=?
        '''
        # params = {
        #     'uid': uid,
        #     'case_number': case_n
        # }
        print(f'[SQL Query]: {query}')

        dbreply = cursor.execute(query, (uid, case_n))
        # n_list = []
        # for item in dbreply:
        #     n_list.append(item)
        return dbreply

r = get_rr_case(uid="test_bot Hpq|RR_test_set_ADn_b34b29a2-a831-4e8f-becb-cbb3fedd432b",
                case_n=36)

print(r)

[SQL Query]: 
        SELECT case_number, rr_normal, rr_abnormal, rr_desc 
        FROM rr_answers
        WHERE uid=? AND case_number=?
        


In [13]:
def execute_and_fetchall(query, params=None):
    # Connect to the SQLite database
    conn = sqlite3.connect('example.db')
    cursor = conn.cursor()
    
    # Execute the query with parameters if provided
    if params:
        cursor.execute(query, params)
    else:
        cursor.execute(query)
    
    # Fetch all results
    rows = cursor.fetchall()
    
    # Get column names from the cursor description
    column_names = [description[0] for description in cursor.description]
    
    # Convert rows to list of dictionaries
    result = [dict(zip(column_names, row)) for row in rows]
    
    # Close the cursor and connection
    cursor.close()
    conn.close()
    
    return result

# Example usage:
query = '''
SELECT case_number, rr_normal, rr_abnormal, rr_desc 
FROM rr_answers
WHERE uid=? AND case_number=?
'''
params = ("test_bot Hpq|RR_test_set_ADn_b34b29a2-a831-4e8f-becb-cbb3fedd432b", 36)
results = execute_and_fetchall(query, params)

# Print the results
for row in results:
    print(row)


(('case_number', None, None, None, None, None, None),
 ('rr_normal', None, None, None, None, None, None),
 ('rr_abnormal', None, None, None, None, None, None),
 ('rr_desc', None, None, None, None, None, None))

In [2]:
connection.close()

In [94]:
query = '''
SELECT uid, COUNT(case_number) FROM rr_answers
WHERE uid='test_bot THJ|RR_test_set_knS_c6aecb6f-619a-4596-8255-bc3a9ee2871b';
'''
reply = cursor.execute(query)

ProgrammingError: Cannot operate on a closed database.

In [74]:
for item in reply.fetchall():
    print(item)

(None, 0)


In [87]:
query = '''
SELECT set_type, finalised, set_name, username, device_name, start_dt FROM sessions;'''
# WHERE uid='test_bot THJ|RR_test_set_knS_c6aecb6f-619a-4596-8255-bc3a9ee2871b';

u_uid_reply = cursor.execute(query)

for i, item in enumerate(u_uid_reply):
    print(f'{i}: {item}')


In [50]:
from collections import namedtuple
def exists_rr_case(
                  rr_case,
                  uid:str,
                  txt:str=None
                  ):
    '''
    Checks for the existance of a single case at a time
    True if exists in te DB
    '''
    cases_in_DB = get_rr_casen_by_uid(uid)
    if rr_case.case_n in cases_in_DB:
        return True
    else:
        return False

uid = 'a311d891-400a-4732-9b21-2f5a9f9dc05f'
rr_case = namedtuple('rr_case', ['case_n', 'uid'])
myrr_case = rr_case(case_n=1, uid=uid)

for i in range(-5, 7):
    myrr_case = rr_case(case_n=i, uid=uid)
    test = exists_rr_case(myrr_case, uid)
    print(f'{i}: {test}')

-5: False
-4: False
-3: False
-2: False
-1: False
0: False
1: True
2: True
3: True
4: True
5: True
6: False


In [88]:
def get_unique_uids(table='sessions'):
    '''
    Queries 'table' for unique UIDs based on type
    table can only be 'sessions', 'rr_answers' or 'lc_answers'
    '''
    if table not in ['sessions', 'rr_answers' or 'lc_answers']:
        raise(ValueError(f"Unknown table: {table}"))
    query = f'''
    SELECT DISTINCT uid FROM {table};
    '''
    reply = cursor.execute(query)
    uids = []
    for item in reply:
        uids.append(item[0])
    return uids

def get_rr_casen_by_uid(uid):
    query = '''
    SELECT case_number FROM rr_answers
    WHERE uid=?'''
    reply = cursor.execute(query, (uid,))
    n_list = []
    for item in reply:
        n_list.append(item[0])
    return n_list

for uid in get_unique_uids(table='rr_answers'):
    uids = get_rr_casen_by_uid(uid)
    print(f'{uid}: {uids}')


In [90]:
def execute_param_query(query, params, txt=''):
    try:
        result = cursor.execute(query, params)
        connection.commit()
        print(f"\t[DB PExecute] '{txt}' Query successful")
    except Exception as e:
        print(f"\t[DB PExecute] The error '{e}' occurred during '{txt}'\n{query} | {params}")
        return e
    return result


In [91]:
from serverconstants import (
    RR_NORMAL,
    RR_ABNORMAL,
    RR_DESC
)
def get_answers_obj_by_uid(uid):
    '''
    Check type - RR or LC
    get list of answers accordingly and populate the answers object
    '''
    # ### Query sessions table:
    query = ("SELECT set_type, finalised, set_name, username, device_name, start_dt"
             " FROM sessions"
             " WHERE uid=?")
    params = (uid, )
    msg = f'Extracting session details for {uid}'
    results = execute_param_query(query, params, txt=msg).fetchall()

    answers = {'type': results[0][0],
        'final': results[0][1],
        'set_id': 'SET-ID',
        'set_name': results[0][2],
        'candidateID': results[0][3],
        'device_name:': results[0][4],
        'start_time': results[0][5],
        'case': {}
    }

    # ### Query rr_answers table:
    query = ("SELECT case_number, rr_normal, rr_abnormal, rr_desc"
             " FROM rr_answers"
             " WHERE uid=?")
    params = (uid, )
    msg = f'Extracting rr answers for {uid}'

    case_results = execute_param_query(query, params, txt=msg).fetchall()

    for item in case_results:
        answers['case'][item[0]] = {
            RR_NORMAL: item[1],
            RR_ABNORMAL: item[2],
            RR_DESC: item[3]
        }
    
    return answers

r, r2, a = get_answers_obj_by_uid(uid='stoat')


	[DB PExecute] 'Extracting session details for stoat' Query successful


IndexError: list index out of range

In [92]:
a

NameError: name 'a' is not defined

In [47]:
from datetime import datetime, timezone 
datetime.now(timezone.utc).astimezone(),timezone.fromutc() # .tzname()

TypeError: unbound method timezone.fromutc() needs an argument

In [48]:
import requests
from datetime import datetime, timezone
import random
import string
import os

charslist = string.ascii_letters + string.digits


SERVER_IP = "http://127.0.0.1:8000"

# ----------------------------------------  New session/GET UID
# @app.post("/session/new/")
# Create new sesion by submitting session details.
# returns uid
route = "/session/new/"
dt_now = datetime.now()
payload = {
  "username": "test_bot " + "".join(random.choices(charslist, k=3)),
  "set_name": "test_set_" + "".join(random.choices(charslist, k=3)),
  "set_type": "RR",
  "device_name": os.environ['COMPUTERNAME'],
  "start_dt": dt_now.isoformat("#", "auto")
}

response = requests.post(SERVER_IP + route, json=payload)

In [57]:
print(payload)

print(response.status_code)
print(type(response.status_code))

print(response.json()['uid'])

print(response.text)

{'username': 'test_bot odh', 'set_name': 'test_set_3e7', 'set_type': 'RR', 'device_name': 'SURFACE3OLI', 'start_dt': '2025-01-23#08:29:07.832364'}
200
<class 'int'>
1a25cb39-a73e-4e38-b832-8738e6afda4b
{"uid":"1a25cb39-a73e-4e38-b832-8738e6afda4b"}


In [69]:
route = "/debug/"
response = requests.get(SERVER_IP + route)
if response.status_code == 200:
    print(f'{route} {response.text}')
else:
    print(f'{route} Request error- status code {response.status_code}')
    print(response.text)
    raise requests.exceptions.HTTPError(response.text)

/debug/ {"hostname":"Surface3Oli","ip":"192.168.49.191"}


In [67]:

dt_now = datetime.now()

j = dt_now.isoformat("#", "auto")
j

'2025-01-23#16:50:16.598495'

In [1]:
data = [{'case_number': 37}, {'case_number': 37}, {'case_number': 1}, {'case_number': 2}, {'case_number': 3}, {'case_number': 4}, {'case_number': 5}, {'case_number': 6}, {'case_number': 7}, {'case_number': 8}, {'case_number': 9}, {'case_number': 10}, {'case_number': 11}, {'case_number': 12}, {'case_number': 13}, {'case_number': 14}, {'case_number': 15}, {'case_number': 16}, {'case_number': 17}, {'case_number': 18}, {'case_number': 19}, {'case_number': 20}, {'case_number': 21}, {'case_number': 22}, {'case_number': 23}, {'case_number': 24}, {'case_number': 25}, {'case_number': 26}, {'case_number': 27}, {'case_number': 28}, {'case_number': 29}, {'case_number': 30}]

existing = []
for item in data:
    existing.append(item['case_number'])

print(existing)


[37, 37, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]


In [1]:
import serverreport as report

report.generate_fake_lc_answers()


{'type': 'LC',
 'set_id': 40,
 'set_name': 'RR_set_70',
 'candidateID': 'Zara Dorsey',
 'device_name': 'mac3',
 'start_time': '20220123-194728.10259',
 'case': {1: {'LC_OBS': 'These are my observations',
   'LC_INT': 'My interpretation...',
   'LC_PDX': 'Prime Differential::',
   'LC_DDX': 'Other differentials...',
   'LC_MX': 'Further Management goes here.'},
  2: {'LC_OBS': '--Intentionally Blank--',
   'LC_INT': '-- left blank --',
   'LC_PDX': '-- left blank --',
   'LC_DDX': '-- left blank --',
   'LC_MX': '-- left blank --'},
  3: {'LC_OBS': 'left sided fracture',
   'LC_INT': 'Foreign Body',
   'LC_PDX': 'Malig Obstruction.',
   'LC_DDX': '1. Infarct\n2. IO\n3.Malignancy',
   'LC_MX': '1. Call someone.\n2. MDM\n3. Surgical referal (urgent)4. CT/MRI follow up.\n5. Onc ref.\n6. Neuro ref for tremor.'},
  4: {'LC_OBS': 'left sided fracture',
   'LC_INT': 'Foreign Body',
   'LC_PDX': 'Pathologic Fracture',
   'LC_DDX': '1. Infarct\n2. IO\n3.Malignancy',
   'LC_MX': '1. Call someone.

In [7]:
msg = {"detail":[{"type":"missing","loc":["body","candidateID"],"msg":"Field required","input":{"uid":"test_bot_jyg|RR_test_set_chQ_fef3129b-d150-4b4c-9b04-b3d5e8fa9de0","case_n":8,"LC_OBS":"my observations text. [UPDATE METHOD]","LC_INT":"my observations text. [UPDATE METHOD]","LC_PDX":"my primary diagnosis text. [UPDATE METHOD]","LC_DDX":"my differentials text. [UPDATE METHOD]","LC_MX":"my management text. [UPDATE METHOD]"}},{"type":"missing","loc":["body","device_name"],"msg":"Field required","input":{"uid":"test_bot_jyg|RR_test_set_chQ_fef3129b-d150-4b4c-9b04-b3d5e8fa9de0","case_n":8,"LC_OBS":"my observations text. [UPDATE METHOD]","LC_INT":"my observations text. [UPDATE METHOD]","LC_PDX":"my primary diagnosis text. [UPDATE METHOD]","LC_DDX":"my differentials text. [UPDATE METHOD]","LC_MX":"my management text. [UPDATE METHOD]"}},{"type":"missing","loc":["body","start_time"],"msg":"Field required","input":{"uid":"test_bot_jyg|RR_test_set_chQ_fef3129b-d150-4b4c-9b04-b3d5e8fa9de0","case_n":8,"LC_OBS":"my observations text. [UPDATE METHOD]","LC_INT":"my observations text. [UPDATE METHOD]","LC_PDX":"my primary diagnosis text. [UPDATE METHOD]","LC_DDX":"my differentials text. [UPDATE METHOD]","LC_MX":"my management text. [UPDATE METHOD]"}},{"type":"missing","loc":["body","set_name"],"msg":"Field required","input":{"uid":"test_bot_jyg|RR_test_set_chQ_fef3129b-d150-4b4c-9b04-b3d5e8fa9de0","case_n":8,"LC_OBS":"my observations text. [UPDATE METHOD]","LC_INT":"my observations text. [UPDATE METHOD]","LC_PDX":"my primary diagnosis text. [UPDATE METHOD]","LC_DDX":"my differentials text. [UPDATE METHOD]","LC_MX":"my management text. [UPDATE METHOD]"}},{"type":"missing","loc":["body","case"],"msg":"Field required","input":{"uid":"test_bot_jyg|RR_test_set_chQ_fef3129b-d150-4b4c-9b04-b3d5e8fa9de0","case_n":8,"LC_OBS":"my observations text. [UPDATE METHOD]","LC_INT":"my observations text. [UPDATE METHOD]","LC_PDX":"my primary diagnosis text. [UPDATE METHOD]","LC_DDX":"my differentials text. [UPDATE METHOD]","LC_MX":"my management text. [UPDATE METHOD]"}},{"type":"missing","loc":["body","type"],"msg":"Field required","input":{"uid":"test_bot_jyg|RR_test_set_chQ_fef3129b-d150-4b4c-9b04-b3d5e8fa9de0","case_n":8,"LC_OBS":"my observations text. [UPDATE METHOD]","LC_INT":"my observations text. [UPDATE METHOD]","LC_PDX":"my primary diagnosis text. [UPDATE METHOD]","LC_DDX":"my differentials text. [UPDATE METHOD]","LC_MX":"my management text. [UPDATE METHOD]"}}]}

print(msg["detail"])

for item in msg["detail"]:
    # for key2 in msg[key]:
    print(item)

[{'type': 'missing', 'loc': ['body', 'candidateID'], 'msg': 'Field required', 'input': {'uid': 'test_bot_jyg|RR_test_set_chQ_fef3129b-d150-4b4c-9b04-b3d5e8fa9de0', 'case_n': 8, 'LC_OBS': 'my observations text. [UPDATE METHOD]', 'LC_INT': 'my observations text. [UPDATE METHOD]', 'LC_PDX': 'my primary diagnosis text. [UPDATE METHOD]', 'LC_DDX': 'my differentials text. [UPDATE METHOD]', 'LC_MX': 'my management text. [UPDATE METHOD]'}}, {'type': 'missing', 'loc': ['body', 'device_name'], 'msg': 'Field required', 'input': {'uid': 'test_bot_jyg|RR_test_set_chQ_fef3129b-d150-4b4c-9b04-b3d5e8fa9de0', 'case_n': 8, 'LC_OBS': 'my observations text. [UPDATE METHOD]', 'LC_INT': 'my observations text. [UPDATE METHOD]', 'LC_PDX': 'my primary diagnosis text. [UPDATE METHOD]', 'LC_DDX': 'my differentials text. [UPDATE METHOD]', 'LC_MX': 'my management text. [UPDATE METHOD]'}}, {'type': 'missing', 'loc': ['body', 'start_time'], 'msg': 'Field required', 'input': {'uid': 'test_bot_jyg|RR_test_set_chQ_fef3

In [16]:
lc_answers = report.generate_fake_lc_answers()
session_uid = 'some_UID'

# Prepare additional info to match pydantic LC_ANS model
lc_answers['uid'] = session_uid

# Fix missing data in the fake case data
for case_n in lc_answers['case'].keys():
    case = lc_answers['case'][case_n]
    case['case_n'] = case_n
    case['uid'] = session_uid

# print(lc_answers)

for key in lc_answers:
    print(f'{key}: {lc_answers}')


for key in lc_answers['case'][1]:
    print(f'{key}: {lc_answers["case"][1][key]}')


type: {'type': 'LC', 'set_id': 79, 'set_name': 'RR_set_82', 'candidateID': 'Jaelynn Lynn', 'device_name': 'mac1', 'start_time': '20220123-135754.751159', 'case': {1: {'LC_OBS': 'These are my observations', 'LC_INT': 'My interpretation...', 'LC_PDX': 'Prime Differential::', 'LC_DDX': 'Other differentials...', 'LC_MX': 'Further Management goes here.', 'case_n': 1, 'uid': 'some_UID'}, 2: {'LC_OBS': '--Intentionally Blank--', 'LC_INT': '-- left blank --', 'LC_PDX': '-- left blank --', 'LC_DDX': '-- left blank --', 'LC_MX': '-- left blank --', 'case_n': 2, 'uid': 'some_UID'}, 3: {'LC_OBS': 'right sided FB', 'LC_INT': 'Intestinal Obstruction', 'LC_PDX': 'Pathologic Fracture', 'LC_DDX': '1. Infarct\n2. IO\n3.Malignancy', 'LC_MX': '1. Call someone.\n2. MDM\n3. Surgical referal (urgent)4. CT/MRI follow up.\n5. Onc ref.\n6. Neuro ref for tremor.', 'case_n': 3, 'uid': 'some_UID'}, 4: {'LC_OBS': 'right sided FB', 'LC_INT': 'Intestinal Obstruction', 'LC_PDX': 'Malig Obstruction.', 'LC_DDX': '1. Inf